In [2]:
import pandas as pd

X = pd.read_csv('../data/secom.data', sep=' ', header=None)
y = pd.read_csv('../data/secom_labels.data', sep=' ', header=None,
                names=['label', 'time'])

In [ ]:
# 센서 590개가 여러가지 센서로 숫자를 잰다. 그 후 합격/불량 판별
X.head()

,0,1,2,3,4,5,6,7,8,9,...,580,581,582,583,584,585,586,587,588,589
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432


In [ ]:
y.head() # -1:정상, 1:불량

,label,time
0,-1,19/07/2008 11:55:00
1,-1,19/07/2008 12:32:00
2,1,19/07/2008 13:17:00
3,-1,19/07/2008 14:43:00
4,-1,19/07/2008 15:22:00


In [11]:
print('X.shape: ', X.shape)
print('y.shape: ', y.shape)

X.shape:  (1567, 590)
y.shape:  (1567, 2)


In [ ]:
y['label'].value_counts() # 정상 1463개, 불량 1개

label
-1    1463
 1     104
Name: count, dtype: int64

In [ ]:
y['label'].value_counts(normalize=True)
# normalize=True: 전체를 1로 봤을때의 비율

# 결과: 불량이 약 6.6%

label
-1    0.933631
 1    0.066369
Name: proportion, dtype: float64

In [ ]:
X.isna().sum().head()
# 비어있는(True)것을 1로 보고 합함

0     6
1     7
2    14
3    14
4    14
dtype: int64

In [17]:
X.isna().sum().sort_values(ascending=False).head(5)

292    1429
293    1429
158    1429
157    1429
492    1341
dtype: int64

In [18]:
X.isna().mean().sort_values(ascending=False).head(5)

292    0.911934
293    0.911934
158    0.911934
157    0.911934
492    0.855775
dtype: float64

In [ ]:
# mean(): 비율, 절반보다 많이 비었는가
(X.isna().mean() > 0.5).sum()

# 센서 28개가 절반넘게 비어 있음

np.int64(28)

In [29]:
# nunique(): 센서(열)에서 서로 다른 값이 몇개인지
X.nunique()

0      1520
1      1504
2       507
3       518
4       503
       ... 
585    1502
586     322
587     260
588     120
589     611
Length: 590, dtype: int64

In [ ]:
# 센서(열)마다 값이 1종류인것 개수
(X.nunique() == 1).sum()

# 센서 116개가 모든 칩에서 값이 같음

np.int64(116)

In [27]:
same = X.columns[X.nunique() == 1]
print(list(same)[:5])

[5, 13, 42, 49, 52]


In [28]:
X[5].unique()

array([100.,  nan])

In [30]:
y["time"] = pd.to_datetime(y["time"].str.strip('"'), format="%d/%m/%Y %H:%M:%S")

In [32]:
y['time']

0      2008-07-19 11:55:00
1      2008-07-19 12:32:00
2      2008-07-19 13:17:00
3      2008-07-19 14:43:00
4      2008-07-19 15:22:00
               ...        
1562   2008-10-16 15:13:00
1563   2008-10-16 20:49:00
1564   2008-10-17 05:26:00
1565   2008-10-17 06:01:00
1566   2008-10-17 06:07:00
Name: time, Length: 1567, dtype: datetime64[ns]

In [33]:
y['time'].min(), y['time'].max()

(Timestamp('2008-07-19 11:55:00'), Timestamp('2008-10-17 06:07:00'))

In [ ]:
# 시간이 계속 늘어나는 순서로 되어있는지
y['time'].is_monotonic_increasing

True

In [37]:
n = int(len(X) * 0.8)
n

1253

In [38]:
X_train, X_test = X.iloc[:n], X.iloc[n:]
y_train, y_test = y.iloc[:n], y.iloc[n:]

In [39]:
X_train.shape, X_test.shape

((1253, 590), (314, 590))

In [40]:
y_train['label'].value_counts()

label
-1    1166
 1      87
Name: count, dtype: int64

In [41]:
y_test['label'].value_counts()

label
-1    297
 1     17
Name: count, dtype: int64

In [42]:
# 값이 한 종류인 센서
const_cols = X_train.columns[X_train.nunique() == 1]
# 절반 넘게 비어있는 센서
miss_cols = X_train.columns[X_train.isna().mean() > 0.5]

In [43]:
len(const_cols), len(miss_cols)

(122, 24)

In [45]:
drop_cols = set(const_cols) | set(miss_cols)
len(drop_cols)

146

In [46]:
X_train2 = X_train.drop(columns=list(drop_cols))
X_test2 = X_test.drop(columns=list(drop_cols))

In [47]:
X_train2.shape, X_test2.shape

((1253, 444), (314, 444))

In [48]:
all_const = set(X.columns[X.nunique() == 1])
diff = set(const_cols) - all_const
len(diff), sorted(diff)

(6, [74, 206, 209, 342, 347, 478])

In [49]:
for c in sorted(diff):
    print(c, X_train[c].unique(), X_test[c].unique())

74 [ 0. nan] [0.     4.1955]
206 [ 0. nan] [0. 2.]
209 [ 0. nan] [ 0.   46.15]
342 [ 0. nan] [0.     0.4472]
347 [ 0. nan] [ 0.     13.9147]
478 [ 0. nan] [  0. 200.]


In [50]:
median = X_train2.median()
X_train2 = X_train2.fillna(median)
X_test2 = X_test2.fillna(median)

In [51]:
# isna().sum(): 빈칸(True)의 합
# isna().sum().sum(): 표 전체의 빈칸(True)의 합

X_train2.isna().sum().sum(), X_test2.isna().sum().sum()

(np.int64(0), np.int64(0))

In [52]:
from sklearn.dummy import DummyClassifier

y_tr = y_train['label']
y_te = y_test['label']

dummy = DummyClassifier(strategy='most_frequent') # 가장 자주 나온 답
dummy.fit(X_train2, y_tr) # 제일 많이 나온 답만 본다
pred_dummy = dummy.predict(X_test2)

print('점수: ', (pred_dummy == y_te).mean())
pd.crosstab(y_te, pred_dummy)

점수:  0.945859872611465


col_0,-1
label,
-1,297
1,17


In [53]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                            random_state=42, n_jobs=-1)
# n_estimators: 300개의 평가 중 다수결로
# class_weight='balanced':  드문 불량을 더 중요하게 보도록 부탁


rf.fit(X_train2, y_tr)
pred_rf = rf.predict(X_test2)

print('점수: ', (pred_rf == y_te).mean())
pd.crosstab(y_te, pred_rf)

점수:  0.945859872611465


col_0,-1
label,
-1,297
1,17
